# Mixture of Experts (MoE)
"Outrageously Large Neural Networks" (Shazeer et al., 2017), Switch Transformer (Fedus et al., 2021), Mixtral (Jiang et al., 2024)

**Idea:** replace each block's FFN with $E$ expert FFNs and a **router** that sends each token to only the top-$k$ of them. Total parameters grow $E\times$, but compute per token only grows $k\times$ (**sparse** activation).

### MoE vs dense SLM
| | Dense SLM (`slm.ipynb`) | MoE |
|---|---|---|
| FFN per block | 1 SwiGLU | $E = 8$ SwiGLU experts + router |
| Params used per token | all | attention + $k = 2$ of 8 experts ("active params") |
| Total params | $P$ | $\approx$ attention + $8\times$ FFN |
| Extra loss | - | load-balancing loss so all experts get used |
| Attention, norm, RoPE | same | same |
| Real examples | Llama, Phi | Mixtral 8x7B, DeepSeek-V3, Qwen-MoE, (reportedly) GPT-4 |

Shape flow: `idx (B, T) -> embed (B, T, D) -> [attn -> router picks 2 of 8 experts per token -> weighted sum] x L -> logits (B, T, V)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - tiny Shakespeare (character level)
~1.1M characters, vocab = 65 unique characters. Each training example is a random chunk of `block_size` characters; the target is the same chunk shifted by one (predict the next character).

The same data and loss are used in `rnn`, `lstm`, `gru`, `slm`, `moe`, `ssm`, so their val losses can be compared.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))          # (1115394,)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 128   # T: characters per sequence
batch_size = 32   # B

def get_batch(split):
    # -> x (B, T), y (B, T); y is x shifted left by one
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

print("vocab size:", vocab_size, "| train chars:", len(train_data), "| val chars:", len(val_data))

## 2. Shared pieces (same as `slm.ipynb`)
RMSNorm, RoPE, causal attention and SwiGLU are copied unchanged; see `slm.ipynb` for their formulas. Each expert below is just one `SwiGLU`.

In [ ]:
class RMSNorm(nn.Module):
    # (..., D) -> (..., D)
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # gamma

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def rope_cache(T, head_dim, base=10000):
    # -> cos, sin each (T, head_dim/2)
    freqs = 1.0 / base ** (torch.arange(0, head_dim, 2).float() / head_dim)  # (head_dim/2,) theta_i
    angles = torch.arange(T).float()[:, None] * freqs[None]                  # (T, head_dim/2) m * theta_i
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    # x (B, heads, T, hd) -> (B, heads, T, hd); rotate each pair (x_2i, x_2i+1)
    x1, x2 = x[..., 0::2], x[..., 1::2]                                      # each (B, heads, T, hd/2)
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


class CausalSelfAttention(nn.Module):
    # (B, T, D) -> (B, T, D)
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, cos, sin):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)  # (B, T, 3, heads, hd)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)                                # each (B, heads, T, hd)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)             # position enters here, not at the input

        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5              # (B, heads, T, T)
        mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()   # (T, T) lower triangle = allowed
        att = att.masked_fill(~mask, float("-inf")).softmax(dim=-1)

        out = (att @ v).transpose(1, 2).reshape(B, T, D)                    # concat heads -> (B, T, D)
        return self.proj(out)


class SwiGLU(nn.Module):
    # (..., D) -> (..., D)
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False)  # gate
        self.w3 = nn.Linear(dim, hidden, bias=False)  # up
        self.w2 = nn.Linear(hidden, dim, bias=False)  # down

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))  # (..., H) -> (..., D)


def ffn_hidden(dim):
    return 64 * ((8 * dim // 3 + 63) // 64)  # 8/3 * D rounded up to a multiple of 64

## 3. MoE layer
Every token is routed **independently**: flatten `(B, T)` into one list of tokens, score the experts, keep the top 2, run each expert only on its tokens, and add the outputs weighted by the router.

### Formula: Router
$$p(x) = \text{softmax}(W_r x) \in \mathbb{R}^{E}, \qquad \mathcal{T}(x) = \text{TopK}(p(x), k)$$
- $x \in \mathbb{R}^{D}$: one token
- $W_r \in \mathbb{R}^{E \times D}$: router weights (`self.router`), $E = 8$ experts
- $p_i(x)$: how much the router likes expert $i$ for this token
- $\mathcal{T}(x)$: indices of the $k = 2$ highest-scoring experts

### Formula: Sparse mixture
$$y = \sum_{i \in \mathcal{T}(x)} g_i(x)\, \text{FFN}_i(x), \qquad g_i(x) = \frac{p_i(x)}{\sum_{j \in \mathcal{T}(x)} p_j(x)}$$
- $\text{FFN}_i$: expert $i$ (a SwiGLU)
- $g_i$: router probability renormalized over the chosen $k$ (Mixtral style), so weights sum to 1
- only $k$ experts run per token; the other $E - k$ cost nothing

### Formula: Load-balancing loss (Switch Transformer)
$$\mathcal{L}_{aux} = E \sum_{i=1}^{E} f_i \, P_i, \qquad f_i = \frac{\#\{\text{assignments to } i\}}{kN}, \qquad P_i = \frac{1}{N}\sum_{x} p_i(x)$$
- $N$: number of tokens in the batch ($B \times T$)
- $f_i$: fraction of routing slots that went to expert $i$ (not differentiable)
- $P_i$: average router probability for expert $i$ (differentiable, carries the gradient)
- minimized (value 1) when routing is uniform; without it the router collapses onto a few experts and the rest never train

In [ ]:
class MoE(nn.Module):
    # (B, T, D) -> (B, T, D), aux loss (scalar)
    def __init__(self, dim, hidden, num_experts=8, top_k=2):
        super().__init__()
        self.num_experts, self.top_k = num_experts, top_k
        self.router = nn.Linear(dim, num_experts, bias=False)                  # W_r
        self.experts = nn.ModuleList([SwiGLU(dim, hidden) for _ in range(num_experts)])

    def forward(self, x):
        B, T, D = x.shape
        x = x.reshape(-1, D)                                    # (N, D) with N = B*T tokens
        probs = self.router(x).softmax(dim=-1)                  # (N, E)
        top_p, top_i = probs.topk(self.top_k, dim=-1)           # each (N, k)
        top_p = top_p / top_p.sum(dim=-1, keepdim=True)         # renormalize over the chosen k

        out = torch.zeros_like(x)                               # (N, D)
        for e, expert in enumerate(self.experts):
            token_idx, slot = (top_i == e).nonzero(as_tuple=True)       # tokens that picked expert e
            if token_idx.numel() == 0:
                continue
            y = expert(x[token_idx]) * top_p[token_idx, slot, None]     # (n_e, D)
            out.index_add_(0, token_idx, y)                             # scatter back to their positions

        f = F.one_hot(top_i, self.num_experts).float().sum(1).mean(0) / self.top_k  # (E,) share of slots
        P = probs.mean(0)                                                           # (E,) mean router prob
        aux = self.num_experts * (f * P).sum()
        self.last_load = f.detach()                             # kept for the usage plot
        return out.reshape(B, T, D), aux

## 4. MoE decoder
The `SLM` from before with `SwiGLU` swapped for `MoE`. The model returns the summed aux loss of all layers alongside the logits.

In [ ]:
class MoEBlock(nn.Module):
    # (B, T, D) -> (B, T, D), aux
    def __init__(self, dim, num_heads, num_experts, top_k):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = CausalSelfAttention(dim, num_heads)
        self.norm2 = RMSNorm(dim)
        self.moe = MoE(dim, ffn_hidden(dim), num_experts, top_k)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)  # (B, T, D)
        y, aux = self.moe(self.norm2(x))
        return x + y, aux                           # (B, T, D)


class MoELM(nn.Module):
    # idx (B, T) -> logits (B, T, V), aux (scalar)
    def __init__(self, vocab_size, block_size, dim=256, depth=6, num_heads=4, num_experts=8, top_k=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([MoEBlock(dim, num_heads, num_experts, top_k) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.embed.weight
        nn.init.normal_(self.embed.weight, std=0.02)
        cos, sin = rope_cache(block_size, dim // num_heads)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def forward(self, idx, state=None):
        T = idx.shape[1]
        x = self.embed(idx)                               # (B, T, D)
        aux = 0.0
        for block in self.blocks:
            x, a = block(x, self.cos[:T], self.sin[:T])   # (B, T, D)
            aux = aux + a
        return self.head(self.norm(x)), aux               # (B, T, V), scalar

model = MoELM(vocab_size, block_size).to(device)
total = sum(p.numel() for p in model.parameters())
per_expert = sum(p.numel() for p in model.blocks[0].moe.experts[0].parameters())
inactive = len(model.blocks) * (8 - 2) * per_expert      # experts a token does not use
print(f"total params: {total / 1e6:.2f}M | active per token: {(total - inactive) / 1e6:.2f}M")
print(model(torch.zeros(2, 10, dtype=torch.long, device=device))[0].shape)

## 5. Training
Same loop as `slm.ipynb` plus the load-balancing term.

### Formula: Total loss
$$\mathcal{L} = \mathcal{L}_{CE} + \alpha \sum_{\ell=1}^{L} \mathcal{L}_{aux}^{(\ell)}$$
- $\mathcal{L}_{CE}$: next-token cross-entropy (as in every LM notebook)
- $\mathcal{L}_{aux}^{(\ell)}$: load-balancing loss of layer $\ell$, $L = 6$ layers
- $\alpha$: 0.01, small so balancing never dominates the LM objective

### Formula: Gradient clipping (by global norm)
$$g \leftarrow g \cdot \min\!\left(1, \frac{c}{\lVert g \rVert_2}\right)$$
- $g$: all parameter gradients concatenated into one vector
- $\lVert g \rVert_2$: its L2 norm
- $c$: max norm (1.0); rescales the whole gradient, keeps its direction

In [ ]:
max_iters = 3000
eval_every = 500
lr = 1e-3
aux_weight = 0.01
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

def loss_fn(model, x, y):
    logits, aux = model(x)                                                   # (B, T, V), scalar
    ce = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))      # (B*T, V) vs (B*T,) -> scalar
    return ce, aux

@torch.no_grad()
def estimate_loss(model, iters=50):
    model.eval()
    out = {s: sum(loss_fn(model, *get_batch(s))[0].item() for _ in range(iters)) / iters for s in ["train", "val"]}
    model.train()
    return out

for it in range(max_iters + 1):
    if it % eval_every == 0:
        l = estimate_loss(model)
        print(f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f}")
    if it == max_iters:
        break
    x, y = get_batch("train")
    ce, aux = loss_fn(model, x, y)
    loss = ce + aux_weight * aux

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

## 6. Expert usage
Share of routing slots each expert received on one validation batch, per layer. With the aux loss these stay near $1/8 = 0.125$; set `aux_weight = 0` and retrain to see the collapse.

In [ ]:
model.eval()
with torch.no_grad():
    model(get_batch("val")[0])
load = torch.stack([b.moe.last_load for b in model.blocks]).cpu()  # (L, E)

plt.figure(figsize=(6, 3))
plt.imshow(load, cmap="viridis", vmin=0, vmax=load.max())
plt.colorbar(label="share of tokens")
plt.xlabel("expert"); plt.ylabel("layer"); plt.title("router load per layer")
plt.show()

## 7. Generation
Identical to the dense SLM; the router runs per generated token.

### Formula: Sampling with temperature
$$p_i = \frac{e^{z_i / \tau}}{\sum_j e^{z_j / \tau}}, \qquad x_{t+1} \sim \text{Categorical}(p)$$
- $z_i$: logit of character $i$
- $\tau$: temperature (0.8); $\tau < 1$ sharpens (safer), $\tau > 1$ flattens (more random), $\tau \to 0$ = greedy argmax

In [ ]:
@torch.no_grad()
def generate(model, start="\n", n=500, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(start)], device=device)           # (1, t)
    for _ in range(n):
        logits, _ = model(idx[:, -block_size:])                   # re-reads the last block_size chars each step (no KV cache)
        probs = F.softmax(logits[:, -1] / temperature, dim=-1)    # (1, V)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)  # (1, t+1)
    return decode(idx[0].tolist())

print(generate(model))

## Notes
- Compared with `slm.ipynb` (4.8M params), the MoE has ~5.7x the total parameters (27.6M) but only ~1.7x the active parameters per token (8.1M), since each token runs 2 of the 8 experts. Mixtral 8x7B is the same trade: 47B total, 13B active.
- The Python loop over experts is for clarity. Real MoE kernels group tokens by expert and run them as one batched matmul, and shard experts across GPUs (expert parallelism).
- Variants: **shared experts** that every token uses (DeepSeek), capacity limits that drop overflow tokens (Switch), and aux-loss-free balancing with a per-expert bias (DeepSeek-V3).